# System to Attention

Step 1 讓模型針對輸入的資訊回傳

Step 2 讓模型對新聞評分

由GPT-4產生
從gemini 評估的prompt 表現較差的組別
"全面評估一項業務創新計畫的可行性，並探討其對目標市場的影響。"
"仔細驗證新業務計畫的各個部分，並預測其對公司股價的可能影響。"
"深入探討業務創新計畫如何影響投資者的預期，並全面評估其長期效益。"

從gemini 評估的prompt 表現較好的組別
"評估一項業務創新計畫如何改變目標市場的格局，並審慎評量其實施的風險。"
"分析新業務計畫對公司股價的可能影響，並審慎評量其對投資者的吸引力。"
"探討業務創新計畫如何影響投資者的預期回報，並審慎評量其對公司財務狀況的影響。"

In [ ]:
# %pip install langchain_google_genai
# %pip install mplfinance

In [ ]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
import google.generativeai as genai
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                convert_system_message_to_human=True,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

# 設定要測試的股票

In [ ]:
import time
import os
import json

def step1(stock_id, st, et):
    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))
    sorted_news_files = [file for file in sorted_news_files if st <= datetime.strptime(file, '%Y%m%d') <= et]
    file_id = 0
    batch_size = 15

    while file_id < len(sorted_news_files):
        batch = []
        for i in range(batch_size):

            if file_id + i >= len(sorted_news_files):
                break
            with open(folder_path + sorted_news_files[file_id+i], 'r', encoding='utf-8') as f:
                news = f.read()

            # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
            messages = [
                ("system","你是一個厲害的股市操盤手，專長是當日沖消"),
                ("human",
                "評估一項業務創新計畫如何改變目標市場的格局，並審慎評量其實施的風險，並保留相關的資訊。" + news)
            ]
            batch.append(messages)  

        res_s2a = llm.batch(batch)

        out_path = os.path.abspath(os.getcwd()) + "/out_SBP/step1/" + stock_id + "/"
        os.makedirs(out_path, exist_ok=True)
        for i in range(len(res_s2a)):
            with open(out_path + sorted_news_files[file_id+i] + ".txt" , 'w',encoding='utf-8') as f:
                f.write(res_s2a[i].content)
        time.sleep(60)
        file_id += batch_size

def step2(stock_id, st, et):
    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))
    sorted_news_files = [file for file in sorted_news_files if st <= datetime.strptime(file, '%Y%m%d') <= et]
    file_id = 0
    batch_size = 15

    while file_id < len(sorted_news_files):
        batch = []
        for i in range(batch_size):

            if file_id + i >= len(sorted_news_files):
                break
            with open(folder_path + sorted_news_files[file_id+i], 'r', encoding='utf-8') as f:
                news = f.read()

            out_path = os.path.abspath(os.getcwd()) + "/out_SBP/step1/" + stock_id + "/"
            with open(out_path + sorted_news_files[file_id+i] + ".txt", 'r', encoding='utf-8') as f:
                step1_out = f.read()

            
            # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
            messages = [
                ("system","你是一個厲害的股市操盤手，專長是當日沖消"),
                ("human",
                "評估一項業務創新計畫如何改變目標市場的格局，並審慎評量其實施的風險，並保留相關的資訊。" + news),
                ("assistant", step1_out),
                ("human", "請用上述資訊對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市無關則回答#unknown，請盡量不要回答 #unknown"),
            ]
            batch.append(messages)

        res_s2a = llm.batch(batch)

        out_path = os.path.abspath(os.getcwd()) + "/out_SBP/step2/" + stock_id + "/"
        os.makedirs(out_path, exist_ok=True)
        for i in range(len(res_s2a)):
            output_data = {
                "model": model,
                "temperature": temperature,
                "top_p": top_p,
                "seed": seed,
                "raw_data": res_s2a[i].content
            }

            json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)
            with open(out_path + sorted_news_files[file_id+i] + ".json" , "w", encoding="UTF-8") as f:
                f.write(json_content)

        time.sleep(60)
        file_id += batch_size

In [ ]:
# 上市公司股票代號
# 代號 ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514', '1101', '2542', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '1604', '2486', '5269', '1314', '8033', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '6176', '1103', '2609', '2002']
# 對應名稱 ['長榮航', '富邦金', '華南金', '長榮鋼', '雄獅', '晶華', '信邦', '裕融', '台達電', '亞力', '台泥', '興富發', '文曄', '群光', '南亞', '統一超', '中環', '長榮', '新光金', '聲寶', '一詮', '祥碩', '中石化', '遠雄', '奇鋐', '遠百', '好樂迪', '金像電', '寶成', '亞泥', '緯創', '豐達科', '智原', '瑞儀', '嘉泥', '陽明', '中鋼']
from datetime import datetime

start_day = datetime(2023, 1, 1)
end_day = datetime(2023, 12, 31)

stock_ids = ['2880']
# stock_names = ['華南金']

# stock_ids = ['2880', '2211', '2707']
# stock_names = ['華南金', '長榮鋼', '晶華']

# stock_ids = ['2618', '2881', '2880', '2211', '2707']
# stock_names = ['長榮航', '富邦金', '華南金', '長榮鋼', '晶華']

# stock_ids = ['9941', '2308', '1514', '1101', '2542']
# stock_names = ['裕融', '台達電', '亞力', '台泥', '興富發']

# # run step 1
# for idx in range(len(stock_ids)):
#     step1(stock_ids[idx], start_day, end_day)
#     print(f"step1 run stock {stock_ids[idx]} done")

# run step 2
for idx in range(len(stock_ids)):
    step2(stock_ids[idx], start_day, end_day)
    print(f"step2 run stock {stock_ids[idx]} done")